# PAID MOBILE GAME LAUNCH STRATEGY ANALYSIS

## Business Context

Stakeholder là một mobile game studio đang chuẩn bị ra mắt một **game Paid / Premium** trên App Store.

Studio đã quyết định đi theo mô hình **Paid**, nên bài phân tích này không trả lời câu hỏi “nên làm Free hay Paid”. Thay vào đó, bài phân tích tập trung vào:

1. Nên ưu tiên genre nào cho Paid game?
2. Genre nào có dấu hiệu willingness-to-pay tốt?
3. Mức giá nào có tín hiệu tốt trong giai đoạn gần nhất?
4. Paid game thành công thường có đặc điểm gì về rating, rating count, price, localization?
5. Nên benchmark game/developer nào?
6. Launch strategy nên đi theo hướng premium/niche/offline/story game hay brand-led game?

## Final Decision

Output của notebook hỗ trợ stakeholder ra quyết định về:

- Genre ưu tiên.
- Price range phù hợp.
- Benchmark game/developer.
- Localization strategy.
- Launch strategy cho mô hình premium.

# 1. Business Question Clarification

## Stakeholder muốn biết điều gì?

Stakeholder muốn biết nếu studio ra mắt một game Paid thì nên chọn genre, mức giá, benchmark và localization như thế nào để tăng khả năng người dùng sẵn sàng trả tiền.

## SMART Objective

Trong phạm vi dataset App Store Games giai đoạn 2008-2019, xác định các genre, price range, game và developer Paid có tín hiệu tốt nhất trong 3 năm gần nhất của dataset, dựa trên weighted rating, rating count, rating velocity, paid revenue proxy và localization proxy.

## North Star Metric

**Paid Market Confidence**

Tức là mức độ thị trường cho thấy người dùng sẵn sàng trả tiền cho một game có chất lượng đáng tin cậy.

## Proxy Metrics

| Proxy Metric | Ý nghĩa business | Giới hạn |
|---|---|---|
| weighted_rating | Chất lượng đáng tin hơn average rating | Vẫn chỉ dựa trên review |
| rating_count | Proxy cho traction | Không phải số lượt mua |
| rating_velocity | Tốc độ tạo traction | Không có lịch sử rating theo tháng |
| price_usd | Giá bán app | Không có lịch sử đổi giá |
| paid_revenue_proxy | Proxy thô cho commercial potential | Không phải doanh thu thật |
| language_count | Proxy cho localization/global reach | Không biết doanh thu theo quốc gia |
| genre_freshness_rate | Genre còn active gần đây không | Không đo trực tiếp demand |

# 2. Scope & Limitation

## Scope chính

Chỉ phân tích game Paid:

`Price per App (USD) > 0`

Free game chỉ dùng để kiểm tra tổng quan, không dùng làm nhánh recommendation chính.

## Recommendation Window

Dataset kéo dài từ 2008-2019. Khi recommend cho game mới, ưu tiên 3 năm gần nhất trong dataset để tránh bị ảnh hưởng bởi trend giá cũ.

## Limitation

Dataset không có:

- Actual sales
- Revenue thật
- Conversion rate
- Refund rate
- IAP revenue
- Ads revenue
- Retention
- DAU/MAU
- Marketing spend
- Rating count history theo tháng/năm
- Lịch sử thay đổi giá

Vì vậy, các chỉ số như `paid_revenue_proxy`, `rating_velocity`, `paid_success_score` chỉ là proxy hỗ trợ quyết định, không phải kết luận tuyệt đối.

# 3. Import Library & Load Data

Upload file `App Store Games.xlsx` khi chạy trên Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

file_path = "/content/App Store Games.xlsx"

if not os.path.exists(file_path):
    from google.colab import files
    uploaded = files.upload()
    file_path = list(uploaded.keys())[0]

df_raw = pd.read_excel(file_path, sheet_name="App Store Games")

print("Dataset shape:", df_raw.shape)
display(df_raw.head())

# 4. Select Necessary Columns

Chỉ chọn các cột cần thiết cho business request:

| Cột | Lý do sử dụng |
|---|---|
| App ID | Xác định app unique |
| Name | Benchmark game |
| Average User Rating | Rating ban đầu |
| User Rating Count | Proxy cho traction |
| Price per App (USD) | Lọc Paid game và phân tích price range |
| Developer | Benchmark developer |
| Age Rating | Positioning/tệp người chơi |
| Languages | Localization |
| Size in Bytes | Friction tải app |
| Primary Genre | Genre chính |
| Genres | Tách genre cụ thể |
| Release Date | Time trend, freshness, rating velocity |

In [ ]:
rename_map = {
    "App ID": "app_id",
    "Name": "name",
    "Average User Rating": "avg_rating",
    "User Rating Count": "rating_count",
    "Price per App (USD)": "price_usd",
    "Developer": "developer",
    "Age Rating": "age_rating",
    "Languages": "languages",
    "Size in Bytes": "size_bytes",
    "Primary Genre": "primary_genre",
    "Genres": "genres",
    "Release Date": "release_date"
}

data = df_raw.rename(columns=rename_map).copy()

selected_cols = [
    "app_id", "name", "avg_rating", "rating_count", "price_usd",
    "developer", "age_rating", "languages", "size_bytes",
    "primary_genre", "genres", "release_date"
]

data = data[selected_cols].copy()
display(data.head())

# 5. Analysis Plan

| Nhóm phân tích | Metric chính | Output |
|---|---|---|
| Paid game market overview | paid_app_count, median_price, total_rating_count | Table + line chart |
| Price range analysis | price_range_score | Table + bar chart |
| Genre freshness | genre_freshness_rate | Table |
| Genre attractiveness | paid_genre_attractiveness_score | Table + bar chart |
| Opportunity matrix | recent_paid_apps vs paid_revenue_proxy | Table + scatter |
| Benchmark games | paid_success_score | Table + bar chart |
| Localization | language_group, high-paid-signal languages | Tables + bar charts |
| Developer benchmark | developer_benchmark_score | Table |
| Case study | Top paid_success_score game | Table |
| Final recommendation | Top genre, price, benchmark | Summary |

In [ ]:
analysis_plan = pd.DataFrame({
    "analysis_group": [
        "Paid game market overview",
        "Price range analysis",
        "Genre freshness",
        "Genre attractiveness",
        "Opportunity matrix",
        "Benchmark games",
        "Localization",
        "Developer benchmark",
        "Case study",
        "Final recommendation"
    ],
    "business_question": [
        "Paid game market gần đây active như thế nào?",
        "Mức giá nào có tín hiệu tốt?",
        "Genre nào còn active gần đây?",
        "Genre nào phù hợp để launch Paid game?",
        "Genre nào vừa có paid signal tốt vừa không quá crowded?",
        "Paid game nào nên benchmark?",
        "Localization nên chọn lọc hay mở rộng?",
        "Developer nào nên benchmark?",
        "Một Paid game tiêu biểu có đặc điểm gì?",
        "Stakeholder nên hành động gì tiếp theo?"
    ],
    "main_metric": [
        "paid_app_count, median_price, total_rating_count",
        "price_range_score",
        "genre_freshness_rate",
        "paid_genre_attractiveness_score",
        "median_paid_revenue_proxy, recent_paid_apps",
        "paid_success_score",
        "language_count, top languages",
        "developer_benchmark_score",
        "paid_success_score",
        "Top genre, price range, benchmark"
    ]
})

display(analysis_plan)

# 6. Data Cleaning & Feature Engineering

Các bước xử lý:

1. Chuyển numeric columns về dạng số.
2. Chuyển release date về datetime.
3. Xóa duplicate theo `app_id`.
4. Tạo `release_year`.
5. Tạo `price_type`.
6. Tạo `game_genre`.
7. Tạo `language_count`.
8. Tạo `size_mb`.
9. Tạo `price_group`.

In [ ]:
numeric_cols = ["avg_rating", "rating_count", "price_usd", "size_bytes"]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

if pd.api.types.is_numeric_dtype(data["release_date"]):
    data["release_date"] = pd.to_datetime(
        data["release_date"], unit="D", origin="1899-12-30", errors="coerce"
    )
else:
    data["release_date"] = pd.to_datetime(data["release_date"], errors="coerce")

before_dedup = len(data)
duplicate_app_count = data["app_id"].duplicated().sum()

data = data.drop_duplicates(subset=["app_id"]).copy()
data = data.dropna(subset=["app_id", "price_usd", "release_date"]).copy()

data["release_year"] = data["release_date"].dt.year.astype(int)
data["price_type"] = np.where(data["price_usd"].eq(0), "Free", "Paid")

def parse_list(x):
    if pd.isna(x):
        return []
    return [i.strip() for i in str(x).split(",") if i.strip()]

GAME_GENRES = {
    "Action", "Adventure", "Arcade", "Board", "Card", "Casino", "Casual",
    "Family", "Music", "Puzzle", "Racing", "Role Playing", "Simulation",
    "Sports", "Strategy", "Trivia", "Word"
}

def extract_game_genre(genres_text, primary_genre):
    genre_list = parse_list(genres_text)
    for g in genre_list:
        if g in GAME_GENRES:
            return g

    if pd.notna(primary_genre) and primary_genre in GAME_GENRES:
        return primary_genre

    for g in genre_list:
        if g not in ["Games", "Entertainment"]:
            return g

    return np.nan

def parse_languages(x):
    if pd.isna(x):
        return []
    return sorted(set([i.strip().upper() for i in str(x).split(",") if i.strip()]))

def assign_price_group(price):
    if pd.isna(price):
        return np.nan
    elif price <= 0:
        return "Free"
    elif price <= 0.99:
        return "$0.99"
    elif price <= 1.99:
        return "$1.99"
    elif price <= 2.99:
        return "$2.99"
    elif price <= 4.99:
        return "$3.99-$4.99"
    elif price <= 9.99:
        return "$5-$9.99"
    else:
        return "$10+"

data["game_genre"] = data.apply(
    lambda row: extract_game_genre(row["genres"], row["primary_genre"]),
    axis=1
)

data["language_list"] = data["languages"].apply(parse_languages)
data["language_count"] = data["language_list"].apply(len)
data["has_english"] = data["language_list"].apply(lambda x: "EN" in x)
data["size_mb"] = data["size_bytes"] / (1024 ** 2)
data["price_group"] = data["price_usd"].apply(assign_price_group)

display(data.head())

# 7. Data Quality Check

Kiểm tra duplicate, missing value và tỷ trọng Free/Paid trước khi filter Paid game.

In [ ]:
quality_check = pd.DataFrame({
    "metric": [
        "rows_before_dedup",
        "duplicate_app_id_before_cleaning",
        "rows_after_cleaning",
        "unique_apps",
        "min_release_year",
        "max_release_year",
        "missing_avg_rating",
        "missing_rating_count",
        "missing_game_genre",
        "missing_languages"
    ],
    "value": [
        before_dedup,
        duplicate_app_count,
        len(data),
        data["app_id"].nunique(),
        data["release_year"].min(),
        data["release_year"].max(),
        data["avg_rating"].isna().sum(),
        data["rating_count"].isna().sum(),
        data["game_genre"].isna().sum(),
        data["languages"].isna().sum()
    ]
})

display(quality_check)

price_type_summary = (
    data.groupby("price_type")
    .agg(
        app_count=("app_id", "nunique"),
        avg_rating=("avg_rating", "mean"),
        total_rating_count=("rating_count", "sum"),
        median_rating_count=("rating_count", "median"),
        median_price=("price_usd", "median")
    )
    .reset_index()
)

price_type_summary["app_share"] = price_type_summary["app_count"] / price_type_summary["app_count"].sum()

display(price_type_summary)

# 8. Filter Paid Games & Recent Paid Games

Tạo:

- `paid_data`: toàn bộ game Paid.
- `recent_paid`: Paid game trong 3 năm gần nhất của dataset.

Giai đoạn gần nhất giúp recommendation ít bị ảnh hưởng bởi trend giá cũ.

In [ ]:
paid_data = data[data["price_type"].eq("Paid")].copy()

latest_year = int(paid_data["release_year"].max())
recent_start_year = latest_year - 2

recent_paid = paid_data[
    paid_data["release_year"].between(recent_start_year, latest_year)
].copy()

print("Latest year in Paid dataset:", latest_year)
print("Recent analysis window:", recent_start_year, "-", latest_year)
print("Total Paid games:", len(paid_data))
print("Recent Paid games:", len(recent_paid))

display(recent_paid.head())

# 9. Create Core Paid Game KPIs

## Weighted Rating

Không dùng Average Rating đơn thuần để ranking vì game ít review có thể bị bias.

Công thức Bayesian Weighted Rating:

`weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C`

## Rating Velocity

`rating_velocity = rating_count / game_age_years`

Đây là proxy cho tốc độ tạo traction.

## Paid Revenue Proxy

`paid_revenue_proxy = price_usd * rating_count`

Đây **không phải doanh thu thật**, chỉ là proxy rất thô cho commercial potential.

In [ ]:
rated_paid_mask = (
    paid_data["rating_count"].fillna(0).gt(0)
    & paid_data["avg_rating"].notna()
)

C = paid_data.loc[rated_paid_mask, "avg_rating"].mean()
m = paid_data.loc[rated_paid_mask, "rating_count"].quantile(0.75)
m = max(1, m)

def add_paid_game_kpis(df, C, m, latest_year):
    df = df.copy()
    votes = df["rating_count"].fillna(0)
    rating = df["avg_rating"].fillna(C)

    df["weighted_rating"] = (
        (votes / (votes + m)) * rating
        + (m / (votes + m)) * C
    )

    df["game_age_years"] = (latest_year - df["release_year"] + 1).clip(lower=1)
    df["rating_velocity"] = votes / df["game_age_years"]
    df["paid_revenue_proxy"] = df["price_usd"].fillna(0) * votes

    return df

paid_data = add_paid_game_kpis(paid_data, C, m, latest_year)
recent_paid = add_paid_game_kpis(recent_paid, C, m, latest_year)

rated_recent_paid = recent_paid[recent_paid["rating_count"].fillna(0).gt(0)].copy()

print("C - Market average rating for Paid games:", round(C, 2))
print("m - Minimum rating count threshold:", round(m, 0))
print("Recent Paid games with rating count > 0:", len(rated_recent_paid))

display(
    rated_recent_paid[
        [
            "name", "developer", "game_genre", "release_year", "price_usd",
            "price_group", "avg_rating", "rating_count", "weighted_rating",
            "rating_velocity", "paid_revenue_proxy", "language_count"
        ]
    ].head()
)

# 10. Output 1 - Paid Game Market Overview

## Business Question

Thị trường Paid game thay đổi như thế nào trong giai đoạn gần nhất?

## Output

- 1 bảng theo năm.
- 1 line chart số lượng Paid game release theo năm.

In [ ]:
yearly_market = (
    recent_paid.groupby("release_year")
    .agg(
        paid_app_count=("app_id", "nunique"),
        median_price=("price_usd", "median"),
        total_rating_count=("rating_count", "sum")
    )
    .reset_index()
)

yearly_rated = (
    rated_recent_paid.groupby("release_year")
    .agg(
        rated_paid_app_count=("app_id", "nunique"),
        median_rating_velocity=("rating_velocity", "median"),
        median_weighted_rating=("weighted_rating", "median"),
        median_paid_revenue_proxy=("paid_revenue_proxy", "median")
    )
    .reset_index()
)

yearly_market = yearly_market.merge(yearly_rated, on="release_year", how="left")

display(yearly_market)

plt.figure(figsize=(10, 5))
plt.plot(yearly_market["release_year"], yearly_market["paid_app_count"], marker="o")
plt.title("Recent Paid Game Releases by Year")
plt.xlabel("Release Year")
plt.ylabel("Paid App Count")
plt.grid(True, alpha=0.3)
plt.show()

print("Business note: Nếu Paid app count giảm hoặc traction thấp, stakeholder cần thận trọng hơn về genre, price và positioning.")

# 11. Output 2 - Price Range Analysis

## Business Question

Mức giá nào có tín hiệu tốt trong giai đoạn gần nhất?

## Logic

Price range tốt nên có:

- Weighted rating ổn.
- Rating count không quá thấp.
- Rating velocity tốt.
- Paid revenue proxy tương đối tốt.

In [ ]:
price_order = ["$0.99", "$1.99", "$2.99", "$3.99-$4.99", "$5-$9.99", "$10+"]

rated_recent_paid["price_group"] = pd.Categorical(
    rated_recent_paid["price_group"],
    categories=price_order,
    ordered=True
)

price_summary = (
    rated_recent_paid.groupby("price_group", observed=False)
    .agg(
        rated_app_count=("app_id", "nunique"),
        median_price=("price_usd", "median"),
        avg_weighted_rating=("weighted_rating", "mean"),
        median_rating_count=("rating_count", "median"),
        median_rating_velocity=("rating_velocity", "median"),
        median_paid_revenue_proxy=("paid_revenue_proxy", "median"),
        total_paid_revenue_proxy=("paid_revenue_proxy", "sum")
    )
    .reset_index()
)

price_summary = price_summary[price_summary["rated_app_count"] > 0].copy()

def minmax_norm(s):
    s = s.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0)
    if s.max() == s.min():
        return pd.Series(0.5, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

price_summary["log_median_rating_count"] = np.log1p(price_summary["median_rating_count"])
price_summary["log_median_revenue_proxy"] = np.log1p(price_summary["median_paid_revenue_proxy"])

price_summary["weighted_rating_norm"] = minmax_norm(price_summary["avg_weighted_rating"])
price_summary["rating_count_norm"] = minmax_norm(price_summary["log_median_rating_count"])
price_summary["revenue_proxy_norm"] = minmax_norm(price_summary["log_median_revenue_proxy"])
price_summary["rating_velocity_norm"] = minmax_norm(np.log1p(price_summary["median_rating_velocity"]))

price_summary["price_range_score"] = (
    0.30 * price_summary["weighted_rating_norm"]
    + 0.25 * price_summary["rating_count_norm"]
    + 0.25 * price_summary["revenue_proxy_norm"]
    + 0.20 * price_summary["rating_velocity_norm"]
) * 100

price_summary = price_summary.sort_values("price_range_score", ascending=False)

display(
    price_summary[
        [
            "price_group", "rated_app_count", "median_price", "avg_weighted_rating",
            "median_rating_count", "median_rating_velocity",
            "median_paid_revenue_proxy", "price_range_score"
        ]
    ]
)

plt.figure(figsize=(10, 5))
plot_data = price_summary.sort_values("price_range_score", ascending=True)
plt.barh(plot_data["price_group"].astype(str), plot_data["price_range_score"])
plt.title("Paid Game Price Range Score")
plt.xlabel("Price Range Score")
plt.ylabel("Price Group")
plt.tight_layout()
plt.show()

best_price_group = price_summary.iloc[0]
print("Recommended price range to investigate:", best_price_group["price_group"])
print("Business note: Đây là mức giá tham khảo ban đầu, cần kết hợp thêm genre, positioning và competitor benchmark.")

# 12. Output 3 - Genre Freshness Rate

## Business Question

Genre nào còn active gần đây trong nhóm Paid game?

## Metric

`genre_freshness_rate = recent_paid_apps / total_paid_apps`

In [ ]:
genre_freshness = (
    paid_data.groupby("game_genre")
    .agg(
        total_paid_apps=("app_id", "nunique"),
        recent_paid_apps_all=("release_year", lambda x: x.ge(recent_start_year).sum())
    )
    .reset_index()
)

genre_freshness["genre_freshness_rate"] = (
    genre_freshness["recent_paid_apps_all"] / genre_freshness["total_paid_apps"]
)

genre_freshness = genre_freshness.sort_values("genre_freshness_rate", ascending=False)

display(genre_freshness.head(15))

print("Business note: Freshness cao nghĩa là genre còn active, nhưng cần kết hợp weighted rating và paid revenue proxy.")

# 13. Output 4 - Genre Attractiveness for Paid Game

## Business Question

Nếu ra mắt game Paid, nên ưu tiên genre nào?

## Paid Genre Attractiveness Score

| Component | Weight |
|---|---:|
| avg_weighted_rating | 30% |
| median_paid_revenue_proxy | 25% |
| median_rating_velocity | 20% |
| total_rating_count | 15% |
| genre_freshness_rate | 10% |

In [ ]:
genre_recent_count = (
    recent_paid.groupby("game_genre")
    .agg(recent_paid_apps=("app_id", "nunique"))
    .reset_index()
)

genre_recent_quality = (
    rated_recent_paid.groupby("game_genre")
    .agg(
        rated_recent_paid_apps=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        median_rating_count=("rating_count", "median"),
        total_rating_count=("rating_count", "sum"),
        median_rating_velocity=("rating_velocity", "median"),
        median_price=("price_usd", "median"),
        median_paid_revenue_proxy=("paid_revenue_proxy", "median"),
        total_paid_revenue_proxy=("paid_revenue_proxy", "sum"),
        median_language_count=("language_count", "median")
    )
    .reset_index()
)

genre_score_base = (
    genre_recent_count
    .merge(genre_recent_quality, on="game_genre", how="left")
    .merge(genre_freshness, on="game_genre", how="left")
)

min_rated_apps = 10
genre_score = genre_score_base[
    genre_score_base["rated_recent_paid_apps"].fillna(0) >= min_rated_apps
].copy()

if genre_score.empty:
    min_rated_apps = 3
    genre_score = genre_score_base[
        genre_score_base["rated_recent_paid_apps"].fillna(0) >= min_rated_apps
    ].copy()

genre_score["log_total_rating_count"] = np.log1p(genre_score["total_rating_count"].fillna(0))
genre_score["log_median_rating_velocity"] = np.log1p(genre_score["median_rating_velocity"].fillna(0))
genre_score["log_median_revenue_proxy"] = np.log1p(genre_score["median_paid_revenue_proxy"].fillna(0))

genre_score["weighted_rating_norm"] = minmax_norm(genre_score["avg_weighted_rating"])
genre_score["revenue_proxy_norm"] = minmax_norm(genre_score["log_median_revenue_proxy"])
genre_score["rating_velocity_norm"] = minmax_norm(genre_score["log_median_rating_velocity"])
genre_score["total_rating_count_norm"] = minmax_norm(genre_score["log_total_rating_count"])
genre_score["freshness_rate_norm"] = minmax_norm(genre_score["genre_freshness_rate"])

genre_score["paid_genre_attractiveness_score"] = (
    0.30 * genre_score["weighted_rating_norm"]
    + 0.25 * genre_score["revenue_proxy_norm"]
    + 0.20 * genre_score["rating_velocity_norm"]
    + 0.15 * genre_score["total_rating_count_norm"]
    + 0.10 * genre_score["freshness_rate_norm"]
) * 100

genre_score = genre_score.sort_values("paid_genre_attractiveness_score", ascending=False)

genre_attractiveness_table = genre_score[
    [
        "game_genre", "recent_paid_apps", "rated_recent_paid_apps",
        "avg_weighted_rating", "median_price", "median_rating_count",
        "total_rating_count", "median_rating_velocity",
        "median_paid_revenue_proxy", "genre_freshness_rate",
        "paid_genre_attractiveness_score"
    ]
].head(15)

display(genre_attractiveness_table)

plt.figure(figsize=(11, 6))
plot_data = genre_attractiveness_table.sort_values("paid_genre_attractiveness_score", ascending=True)
plt.barh(plot_data["game_genre"], plot_data["paid_genre_attractiveness_score"])
plt.title("Top Genres for Paid Game Launch")
plt.xlabel("Paid Genre Attractiveness Score")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

best_genre = genre_score.iloc[0]
print("Top recommended genre to investigate:", best_genre["game_genre"])
print("Business note: Genre này cần được validate thêm bằng competitor review và willingness-to-pay test.")

# 14. Output 5 - Paid Genre Opportunity Matrix

## Business Question

Genre nào vừa có paid potential tốt vừa không quá crowded?

| Nhóm | Ý nghĩa |
|---|---|
| Priority opportunity | Paid potential cao, cạnh tranh chưa quá cao |
| Big market / high competition | Paid potential cao nhưng cạnh tranh mạnh |
| Crowded / weak paid signal | Nhiều app nhưng paid signal chưa tốt |
| Niche / validate carefully | Quy mô nhỏ hơn, cần test kỹ |

In [ ]:
competition_cut = genre_score["recent_paid_apps"].quantile(0.75)
paid_signal_cut = genre_score["median_paid_revenue_proxy"].quantile(0.75)

def classify_paid_opportunity(row):
    if row["median_paid_revenue_proxy"] >= paid_signal_cut and row["recent_paid_apps"] < competition_cut:
        return "Priority opportunity"
    elif row["median_paid_revenue_proxy"] >= paid_signal_cut and row["recent_paid_apps"] >= competition_cut:
        return "Big market / high competition"
    elif row["median_paid_revenue_proxy"] < paid_signal_cut and row["recent_paid_apps"] >= competition_cut:
        return "Crowded / weak paid signal"
    else:
        return "Niche / validate carefully"

genre_score["opportunity_type"] = genre_score.apply(classify_paid_opportunity, axis=1)

genre_opportunity_table = genre_score[
    [
        "game_genre", "recent_paid_apps", "median_paid_revenue_proxy",
        "median_rating_velocity", "avg_weighted_rating",
        "genre_freshness_rate", "paid_genre_attractiveness_score",
        "opportunity_type"
    ]
].sort_values("paid_genre_attractiveness_score", ascending=False)

display(genre_opportunity_table)

plt.figure(figsize=(10, 6))

plt.scatter(
    genre_score["recent_paid_apps"],
    genre_score["median_paid_revenue_proxy"],
    s=np.sqrt(genre_score["total_rating_count"].clip(lower=1)) * 2,
    alpha=0.6
)

for _, row in genre_score.head(10).iterrows():
    plt.text(row["recent_paid_apps"], row["median_paid_revenue_proxy"], row["game_genre"], fontsize=9)

plt.axvline(competition_cut, linestyle="--", alpha=0.5)
plt.axhline(paid_signal_cut, linestyle="--", alpha=0.5)

plt.title("Paid Genre Opportunity Matrix")
plt.xlabel("Competition Proxy: Recent Paid App Count")
plt.ylabel("Paid Potential Proxy: Median Paid Revenue Proxy")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

priority_genres = genre_opportunity_table[
    genre_opportunity_table["opportunity_type"].eq("Priority opportunity")
]

if len(priority_genres) > 0:
    print("Priority opportunity genres:", ", ".join(priority_genres["game_genre"].head(3).tolist()))
else:
    print("No clear Priority opportunity genre. Consider top-ranked genre but validate carefully.")

# 15. Output 6 - Top Paid Benchmark Games

## Business Question

Paid game nào nên dùng để benchmark?

## Paid Success Score

| Component | Weight |
|---|---:|
| weighted_rating | 30% |
| paid_revenue_proxy | 30% |
| rating_velocity | 20% |
| rating_count | 10% |
| language_count | 10% |

In [ ]:
score_base = rated_recent_paid.copy()

score_base["log_rating_count"] = np.log1p(score_base["rating_count"].fillna(0))
score_base["log_rating_velocity"] = np.log1p(score_base["rating_velocity"].fillna(0))
score_base["log_paid_revenue_proxy"] = np.log1p(score_base["paid_revenue_proxy"].fillna(0))

score_base["weighted_rating_norm"] = minmax_norm(score_base["weighted_rating"])
score_base["revenue_proxy_norm"] = minmax_norm(score_base["log_paid_revenue_proxy"])
score_base["rating_velocity_norm"] = minmax_norm(score_base["log_rating_velocity"])
score_base["rating_count_norm"] = minmax_norm(score_base["log_rating_count"])
score_base["language_count_norm"] = minmax_norm(score_base["language_count"])

score_base["paid_success_score"] = (
    0.30 * score_base["weighted_rating_norm"]
    + 0.30 * score_base["revenue_proxy_norm"]
    + 0.20 * score_base["rating_velocity_norm"]
    + 0.10 * score_base["rating_count_norm"]
    + 0.10 * score_base["language_count_norm"]
) * 100

top_paid_games = (
    score_base.sort_values("paid_success_score", ascending=False)
    [
        [
            "name", "developer", "game_genre", "release_year",
            "price_usd", "price_group", "avg_rating", "rating_count",
            "weighted_rating", "rating_velocity", "paid_revenue_proxy",
            "language_count", "paid_success_score"
        ]
    ]
    .head(20)
)

display(top_paid_games)

plt.figure(figsize=(12, 6))
plot_data = top_paid_games.head(10).sort_values("paid_success_score", ascending=True)
plt.barh(plot_data["name"], plot_data["paid_success_score"])
plt.title("Top Paid Benchmark Games by Success Score")
plt.xlabel("Paid Success Score")
plt.ylabel("Game")
plt.tight_layout()
plt.show()

best_game = top_paid_games.iloc[0]
print("Top benchmark game:", best_game["name"], "-", best_game["developer"])

# 16. Output 7 - Localization Strategy for Paid Games

## Business Question

Localization nên làm rộng hay chọn lọc với Paid game?

Với Paid game, localization có thể giúp mở rộng thị trường nhưng cần cân nhắc chi phí vì người dùng phải trả tiền trước.

In [ ]:
def language_group(n):
    if n <= 1:
        return "1 language"
    elif n <= 5:
        return "2-5 languages"
    elif n <= 10:
        return "6-10 languages"
    else:
        return "11+ languages"

rated_recent_paid["language_group"] = rated_recent_paid["language_count"].apply(language_group)

localization_summary = (
    rated_recent_paid.groupby("language_group", observed=False)
    .agg(
        rated_app_count=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        median_rating_count=("rating_count", "median"),
        median_rating_velocity=("rating_velocity", "median"),
        median_paid_revenue_proxy=("paid_revenue_proxy", "median"),
        total_paid_revenue_proxy=("paid_revenue_proxy", "sum")
    )
    .reset_index()
)

language_order = ["1 language", "2-5 languages", "6-10 languages", "11+ languages"]

localization_summary["language_group"] = pd.Categorical(
    localization_summary["language_group"],
    categories=language_order,
    ordered=True
)

localization_summary = localization_summary.sort_values("language_group")

display(localization_summary)

plt.figure(figsize=(10, 5))
plt.bar(localization_summary["language_group"].astype(str), localization_summary["median_paid_revenue_proxy"])
plt.title("Median Paid Revenue Proxy by Localization Group")
plt.xlabel("Localization Group")
plt.ylabel("Median Paid Revenue Proxy")
plt.tight_layout()
plt.show()

best_lang_group = localization_summary.sort_values("median_paid_revenue_proxy", ascending=False).iloc[0]
print("Best localization group by median paid revenue proxy:", best_lang_group["language_group"])

# 17. Output 8 - Top Languages Among High Paid-Signal Games

## Business Question

Nếu launch Paid game, nên ưu tiên ngôn ngữ nào ở phase đầu?

Lấy nhóm game có `paid_success_score` thuộc top 10%.

In [ ]:
high_paid_signal_cut = score_base["paid_success_score"].quantile(0.90)

high_paid_signal_games = score_base[
    score_base["paid_success_score"] >= high_paid_signal_cut
].copy()

language_counter = {}

for langs in high_paid_signal_games["language_list"]:
    for lang in langs:
        language_counter[lang] = language_counter.get(lang, 0) + 1

top_languages_high_paid_signal = (
    pd.DataFrame(language_counter.items(), columns=["language", "high_paid_signal_game_count"])
    .sort_values("high_paid_signal_game_count", ascending=False)
    .head(15)
)

display(top_languages_high_paid_signal)

plt.figure(figsize=(10, 5))
plot_data = top_languages_high_paid_signal.sort_values("high_paid_signal_game_count", ascending=True)
plt.barh(plot_data["language"], plot_data["high_paid_signal_game_count"])
plt.title("Top Languages Among High Paid-Signal Games")
plt.xlabel("Number of High Paid-Signal Games")
plt.ylabel("Language")
plt.tight_layout()
plt.show()

print("Business note: Dùng các ngôn ngữ này như shortlist cho localization phase 1, không phải kết luận nhân quả.")

# 18. Output 9 - Developer Benchmark

## Business Question

Developer nào nên benchmark nếu studio muốn launch game Paid?

## Developer Benchmark Score

| Component | Weight |
|---|---:|
| avg_weighted_rating | 30% |
| total_paid_revenue_proxy | 30% |
| median_rating_velocity | 20% |
| median_price | 10% |
| median_language_count | 10% |

In [ ]:
developer_summary_base = (
    rated_recent_paid.groupby("developer")
    .agg(
        rated_paid_app_count=("app_id", "nunique"),
        avg_weighted_rating=("weighted_rating", "mean"),
        total_rating_count=("rating_count", "sum"),
        median_rating_count=("rating_count", "median"),
        median_rating_velocity=("rating_velocity", "median"),
        median_price=("price_usd", "median"),
        total_paid_revenue_proxy=("paid_revenue_proxy", "sum"),
        median_paid_revenue_proxy=("paid_revenue_proxy", "median"),
        median_language_count=("language_count", "median"),
        main_genre=("game_genre", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    )
    .reset_index()
)

developer_summary = developer_summary_base[
    developer_summary_base["rated_paid_app_count"] >= 2
].copy()

if developer_summary.empty:
    developer_summary = developer_summary_base[
        developer_summary_base["rated_paid_app_count"] >= 1
    ].copy()

developer_summary["log_total_revenue_proxy"] = np.log1p(developer_summary["total_paid_revenue_proxy"].fillna(0))
developer_summary["log_median_rating_velocity"] = np.log1p(developer_summary["median_rating_velocity"].fillna(0))

developer_summary["weighted_rating_norm"] = minmax_norm(developer_summary["avg_weighted_rating"])
developer_summary["revenue_proxy_norm"] = minmax_norm(developer_summary["log_total_revenue_proxy"])
developer_summary["rating_velocity_norm"] = minmax_norm(developer_summary["log_median_rating_velocity"])
developer_summary["price_norm"] = minmax_norm(developer_summary["median_price"])
developer_summary["language_count_norm"] = minmax_norm(developer_summary["median_language_count"])

developer_summary["developer_benchmark_score"] = (
    0.30 * developer_summary["weighted_rating_norm"]
    + 0.30 * developer_summary["revenue_proxy_norm"]
    + 0.20 * developer_summary["rating_velocity_norm"]
    + 0.10 * developer_summary["price_norm"]
    + 0.10 * developer_summary["language_count_norm"]
) * 100

top_developers = (
    developer_summary.sort_values("developer_benchmark_score", ascending=False)
    [
        [
            "developer", "main_genre", "rated_paid_app_count",
            "avg_weighted_rating", "median_price", "total_rating_count",
            "median_rating_velocity", "total_paid_revenue_proxy",
            "median_language_count", "developer_benchmark_score"
        ]
    ]
    .head(20)
)

display(top_developers)

best_developer = top_developers.iloc[0]
print("Top benchmark developer:", best_developer["developer"])

# 19. Output 10 - Case Study: Best Benchmark Paid Game

Chọn game có `paid_success_score` cao nhất để làm case study.

In [ ]:
case_study_game = score_base.sort_values("paid_success_score", ascending=False).head(1).copy()

case_study_cols = [
    "name", "developer", "game_genre", "release_year", "age_rating",
    "price_usd", "price_group", "avg_rating", "rating_count",
    "weighted_rating", "game_age_years", "rating_velocity",
    "paid_revenue_proxy", "language_count", "languages",
    "size_mb", "paid_success_score"
]

display(case_study_game[case_study_cols])

case = case_study_game.iloc[0]

print("CASE STUDY SUMMARY")
print("-" * 60)
print("Game:", case["name"])
print("Developer:", case["developer"])
print("Genre:", case["game_genre"])
print("Release year:", int(case["release_year"]))
print("Price:", round(case["price_usd"], 2))
print("Price group:", case["price_group"])
print("Weighted rating:", round(case["weighted_rating"], 2))
print("Rating count:", int(case["rating_count"]))
print("Rating velocity:", round(case["rating_velocity"], 0))
print("Paid revenue proxy:", round(case["paid_revenue_proxy"], 0))
print("Language count:", int(case["language_count"]))
print("Paid success score:", round(case["paid_success_score"], 1))

# 20. Final Insight & Recommendation

Phần này tổng hợp output chính để stakeholder ra quyết định.

In [ ]:
top_3_genres = genre_score.head(3)[
    [
        "game_genre", "recent_paid_apps", "rated_recent_paid_apps",
        "median_price", "avg_weighted_rating", "median_paid_revenue_proxy",
        "median_rating_velocity", "genre_freshness_rate",
        "paid_genre_attractiveness_score", "opportunity_type"
    ]
]

top_5_games = top_paid_games.head(5)
top_5_developers = top_developers.head(5)
top_5_languages = top_languages_high_paid_signal.head(5)

top_price_group = price_summary.head(3)[
    [
        "price_group", "rated_app_count", "median_price",
        "avg_weighted_rating", "median_rating_count",
        "median_paid_revenue_proxy", "price_range_score"
    ]
]

print("TOP 3 GENRE CANDIDATES")
display(top_3_genres)

print("TOP 3 PRICE RANGE CANDIDATES")
display(top_price_group)

print("TOP 5 BENCHMARK PAID GAMES")
display(top_5_games)

print("TOP 5 BENCHMARK DEVELOPERS")
display(top_5_developers)

print("TOP 5 LANGUAGES FOR LOCALIZATION PHASE 1")
display(top_5_languages)

best_genre = top_3_genres.iloc[0]
best_price = top_price_group.iloc[0]
best_game = top_5_games.iloc[0]
best_developer = top_5_developers.iloc[0]

phase_1_languages = ", ".join(top_5_languages["language"].tolist())

print("\nEXECUTIVE RECOMMENDATION")
print("-" * 80)

print("1. Genre priority:", best_genre["game_genre"])
print("   Paid genre attractiveness score:", round(best_genre["paid_genre_attractiveness_score"], 1))
print("   Opportunity type:", best_genre["opportunity_type"])

print("\n2. Price range to investigate:", best_price["price_group"])
print("   Price range score:", round(best_price["price_range_score"], 1))
print("   Median price:", round(best_price["median_price"], 2))

print("\n3. Benchmark game:", best_game["name"], "-", best_game["developer"])
print("   Benchmark developer:", best_developer["developer"])

print("\n4. Localization phase 1 languages:", phase_1_languages)

print("\n5. Recommended next actions:")
print("   - Validate genre", best_genre["game_genre"], "with competitor review and willingness-to-pay survey.")
print("   - Test price range", best_price["price_group"], "with survey, landing page test, or soft-launch pricing test.")
print("   - Benchmark top Paid games/developers for positioning, perceived value, and launch messaging.")

print("\nImportant limitation:")
print("Recommendations are based on App Store rating proxy metrics, not actual sales, revenue, conversion, refund, or retention data.")

# 21. Report Flow Suggestion

Business request không yêu cầu dashboard, nên không tạo dashboard.

Nếu cần trình bày thành report hoặc slide, dùng flow:

1. Executive Summary
2. Business Question & Dataset Limitation
3. Paid Game Market Overview
4. Price Range Analysis
5. Genre Opportunity
6. Benchmark Game/Developer
7. Localization Strategy
8. Case Study
9. Final Recommendation

## Message chính

> Nếu launch game Paid, stakeholder nên ưu tiên genre có weighted rating tốt, paid revenue proxy ổn, price range hợp lý và còn active trong giai đoạn gần nhất.